# RAG Pipeline — PDF Analysis with LangChain
## End-to-end workflow: ingest PDFs → build vector store → run analysis queries → export CSV + HTML report with charts

### Architecture Overview
```
PDF Files  →  Text Extraction  →  Chunking  →  ChromaDB (Vector Store)
                                                        ↓
                                          RAG Chain (LLM + Retriever)
                                                        ↓
                               Structured Analysis (topics, entities, sentiment)
                                                        ↓
                                  CSV Export  +  Plotly Charts  →  HTML Report
```

### Steps
1. Environment Setup & Package Installation
2. Configuration (paths, API keys, model settings)
3. PDF Ingestion & Text Extraction
4. Text Chunking & Preprocessing
5. Embeddings & ChromaDB Vector Store
6. RAG Chain Construction
7. Batch Document Analysis
8. Results Export to CSV
9. Visualizations with Plotly
10. HTML Report Generation

## Step 1 — Environment Setup & Package Installation

In [20]:
# Install all required packages
# Run once; comment out after first execution
# %pip install -q \
#     langchain==0.2.16 \
#     langchain-community==0.2.16 \
#     langchain-core==0.2.38 \
#     langchain-openai==0.1.23 \
#     langchain-cohere==0.1.9 \
#     chromadb==0.5.5 \
#     sentence-transformers==3.0.1 \
#     pypdf==4.3.1 \
#     pandas==2.2.2 \
#     plotly==5.22.0 \
#     "nbformat>=4.2.0" \
#     jinja2==3.1.4 \
#     python-dotenv==1.0.1 \
#     openpyxl==3.1.5 \
#     kaleido==0.2.1

## Step 2 — Configuration

Set your PDF folder path, API keys, model choices, and chunking parameters here.
All subsequent cells read from this configuration block.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads .env file if present

# ── Paths ──────────────────────────────────────────────────────────────────
PDF_DIR         = Path("./pdfs")            # folder containing your PDF files
VECTORSTORE_DIR = Path("./vectorstore")     # ChromaDB persistence directory
OUTPUT_DIR      = Path("./output")          # CSV + HTML output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUTPUT      = OUTPUT_DIR / "analysis_results.csv"
HTML_REPORT     = OUTPUT_DIR / "report.html"

# ── LLM Provider ───────────────────────────────────────────────────────────
# Options: "openai" | "cohere"
LLM_PROVIDER    = "openai"
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "sk-...")
COHERE_API_KEY  = os.getenv("COHERE_API_KEY", "")

# ── Model names ────────────────────────────────────────────────────────────
OPENAI_MODEL    = "gpt-4o-mini"             # or "gpt-3.5-turbo"
COHERE_MODEL    = "command-r"
EMBEDDING_MODEL = "all-mpnet-base-v2"       # HuggingFace sentence-transformer

# ── Chunking ────────────────────────────────────────────────────────────────
CHUNK_SIZE      = 1024
CHUNK_OVERLAP   = 128
TOP_K_RETRIEVAL = 4                          # docs returned per query

# ── Analysis queries — Software Solution Architecture learning materials ───
ANALYSIS_QUERIES = {
    # ── Overview ────────────────────────────────────────────────────────────
    "summary": (
        "Provide a concise 3-sentence summary of this document, "
        "focusing on its purpose within a Software Solution Architecture curriculum."
    ),
    "learning_objectives": (
        "List the key learning objectives or competencies a student should gain "
        "from this material. Return as a numbered list."
    ),
    "module_topic": (
        "What is the primary architectural topic or domain covered by this document? "
        "You MUST pick exactly ONE label from this fixed list — do not invent new labels:\n"
        "  Solution Architecture, Microservices, Event-Driven Architecture, "
        "Cloud-Native Design, API Design, Security Architecture, "
        "Data Architecture, DevOps & CI/CD, System Design, "
        "Enterprise Architecture, Software Engineering Fundamentals, "
        "Governance & Standards\n"
        "Return only the label, nothing else."
    ),

    # ── Architecture content ────────────────────────────────────────────────
    "architecture_patterns": (
        "List all software architecture patterns, styles, or principles explicitly "
        "mentioned or taught (e.g. CQRS, Saga, Strangler Fig, Hexagonal, "
        "Event Sourcing, 12-Factor). Return as comma-separated values."
    ),
    "technologies_tools": (
        "List all specific technologies, frameworks, cloud services, or tools "
        "referenced (e.g. Kafka, Kubernetes, AWS Lambda, OAuth2, OpenAPI). "
        "Return as comma-separated values."
    ),
    "architectural_decisions": (
        "What key architectural trade-offs, decision criteria, or ADR (Architecture "
        "Decision Record) topics are discussed? Summarise in 2–3 sentences."
    ),

    # ── Pedagogy ────────────────────────────────────────────────────────────
    "difficulty_level": (
        "Assess the difficulty level of this material for a software architect student. "
        "Answer with one word: Introductory, Intermediate, or Advanced."
    ),
    "prerequisites": (
        "What prior knowledge or skills does this material assume the student already has? "
        "List as comma-separated values, or 'None stated' if not mentioned."
    ),
    "exercises_labs": (
        "Are there any practical exercises, labs, case studies, or assignments described? "
        "If yes, briefly describe them. If no, answer 'None'."
    ),

    # ── Quality & completeness ──────────────────────────────────────────────
    "key_concepts": (
        "List the top 7 key architectural concepts or terms defined or explained "
        "in this document. Return as comma-separated values."
    ),
    "gaps_recommendations": (
        "Identify any apparent gaps, outdated content, or areas that could be "
        "strengthened in this learning material from an enterprise architecture perspective."
    ),
}

print("✅ Configuration loaded")
print(f"   PDF source  : {PDF_DIR.resolve()}")
print(f"   Vector store: {VECTORSTORE_DIR.resolve()}")
print(f"   Analysis queries: {len(ANALYSIS_QUERIES)} defined")
print(f"   Output dir  : {OUTPUT_DIR.resolve()}")
print(f"   LLM         : {LLM_PROVIDER} / {OPENAI_MODEL if LLM_PROVIDER == 'openai' else COHERE_MODEL}")


## Step 3 — PDF Ingestion & Text Extraction

`PyPDFLoader` reads each PDF page-by-page and attaches metadata:
- `source` — file path
- `page` — page number (0-indexed)
- `filename` — base filename (used later as document identity)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.schema import Document
from typing import List
import glob

def load_pdfs(pdf_dir: Path) -> List[Document]:
    """Load all PDFs from a directory.  Returns a flat list of page-level Documents."""
    pdf_files = sorted(pdf_dir.glob("*.pdf"))
    
    if not pdf_files:
        raise FileNotFoundError(
            f"No PDF files found in '{pdf_dir.resolve()}'.\n"
            "Please copy your PDF files into that folder and re-run this cell."
        )
    
    all_docs: List[Document] = []
    print(f"Found {len(pdf_files)} PDF file(s) in {pdf_dir.resolve()}\n")
    
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        pages  = loader.load()
        
        # Enrich every page with a clean filename tag
        for page in pages:
            page.metadata["filename"] = pdf_path.name
            page.metadata["filepath"] = str(pdf_path)
        
        all_docs.extend(pages)
        print(f"  [{pdf_path.name}]  → {len(pages)} page(s) loaded")
    
    print(f"\nTotal pages extracted: {len(all_docs)}")
    return all_docs

# Create the PDF folder if it doesn't exist yet
PDF_DIR.mkdir(parents=True, exist_ok=True)

raw_docs = load_pdfs(PDF_DIR)

## Step 4 — Text Chunking & Preprocessing

Split the page-level documents into smaller, overlapping chunks suitable for embedding.
`RecursiveCharacterTextSplitter` tries to break on paragraphs → sentences → words in order,
preserving semantic coherence within each chunk.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len,
)

chunked_docs: List[Document] = splitter.split_documents(raw_docs)

# Add a chunk index to each document for traceability
for idx, doc in enumerate(chunked_docs):
    doc.metadata["chunk_id"] = idx

# Quick stats
import pandas as pd

stats_df = (
    pd.DataFrame([
        {
            "filename": d.metadata.get("filename", "unknown"),
            "page":     d.metadata.get("page", -1),
            "chunk_id": d.metadata.get("chunk_id"),
            "chars":    len(d.page_content),
        }
        for d in chunked_docs
    ])
)

print(f"Total chunks : {len(chunked_docs)}")
print(f"Avg chunk size: {stats_df['chars'].mean():.0f} chars\n")
print("Chunks per document:")
print(stats_df.groupby("filename")["chunk_id"].count().to_string())

## Step 5 — Embeddings & ChromaDB Vector Store

Each chunk is converted to a dense vector with a sentence-transformer model and stored in ChromaDB.
The store is persisted to disk so you can reload it without re-embedding.

> **Tip:** If you change the PDF set or the embedding model, delete the `vectorstore/` folder first.

In [ ]:

import chromadb
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ── 1. Embedding model ────────────────────────────────────────────────────
print(f"Loading embedding model: {EMBEDDING_MODEL} …")
embedding = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model ready.\n")

# ── 2. Persist directory & collection name ────────────────────────────────
persist_dir     = str(VECTORSTORE_DIR.resolve())
collection_name = "rag_docs"
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

# ── 3. Build or reload the vector store ───────────────────────────────────
chroma_client = chromadb.PersistentClient(path=persist_dir)

# Check whether the collection already contains documents
existing   = chroma_client.list_collections()
has_data   = any(c.name == collection_name for c in existing)

if has_data:
    col           = chroma_client.get_collection(collection_name)
    collection_count = col.count()
    has_data      = collection_count > 0

if has_data:
    print(f"Re-using existing ChromaDB collection '{collection_name}' "
          f"({collection_count} vectors).")
    vectordb = Chroma(
        collection_name=collection_name,
        embedding_function=embedding,
        persist_directory=persist_dir,
        client=chroma_client,
    )
else:
    print(f"Building ChromaDB collection '{collection_name}' "
          f"from {len(chunked_docs)} chunks …")
    vectordb = Chroma.from_documents(
        documents=chunked_docs,
        embedding=embedding,
        collection_name=collection_name,
        persist_directory=persist_dir,
        client=chroma_client,
    )
    collection_count = vectordb._collection.count()
    print(f"Stored {collection_count} vectors.\n")

# ── 4. Retriever ──────────────────────────────────────────────────────────
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": TOP_K_RETRIEVAL,
        "fetch_k": TOP_K_RETRIEVAL * 4,
        "lambda_mult": 0.6,
    },
)

print(f"\n✅ Vector store ready  — {collection_count} vectors in '{collection_name}'")
print(f"   Retriever          : MMR, top-{TOP_K_RETRIEVAL}")
print(f"   Persist directory  : {persist_dir}")


## Step 6 — RAG Chain Construction

Builds a `ConversationalRetrievalChain` that:
1. Fetches the top-K most relevant chunks from ChromaDB for each question
2. Passes them as context to the LLM alongside the conversation history
3. Returns a grounded answer

Supports both **OpenAI** and **Cohere** — controlled by `LLM_PROVIDER` in Step 2.

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# ── LLM selection ──────────────────────────────────────────────────────────
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=OPENAI_MODEL,
        openai_api_key=OPENAI_API_KEY,
        temperature=0,
    )
    print(f"LLM: OpenAI {OPENAI_MODEL}")

elif LLM_PROVIDER == "cohere":
    from langchain_cohere import ChatCohere
    llm = ChatCohere(
        model=COHERE_MODEL,
        cohere_api_key=COHERE_API_KEY,
        temperature=0,
    )
    print(f"LLM: Cohere {COHERE_MODEL}")

else:
    raise ValueError(f"Unknown LLM_PROVIDER: '{LLM_PROVIDER}'. Choose 'openai' or 'cohere'.")

# ── Custom QA prompt — allows synthesis, not just extraction ───────────────
QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are an expert assistant for a Software Solution Architecture programme.\n"
        "Use the retrieved document excerpts below to answer the question.\n"
        "Synthesise information across all excerpts when summarising.\n"
        "If the excerpts don't contain enough information, say so briefly but still "
        "share what you can infer from them.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

# ── Chain with conversational memory ──────────────────────────────────────
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
    output_key="answer",
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    verbose=False,
)

print("RAG chain ready.\n")

# ── Quick manual test ──────────────────────────────────────────────────────
test_query = "What architectural topics and key concepts are covered across these documents?"
result = qa_chain({"question": test_query, "chat_history": []})
print(f"Test query  : {test_query}")
print(f"Answer      : {result['answer']}")
print(f"\nSource docs : {[d.metadata.get('filename') for d in result['source_documents']]}")


## Step 7 — Batch Document Analysis

For every unique PDF file, run the structured analysis queries defined in Step 2.
Each query uses a **document-scoped retriever** (MMR search filtered by filename) so answers
relate only to that specific file.

Results are collected into a list of dictionaries for export.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

def analyse_document(filename: str, file_content: str, queries: dict) -> dict:
    """
    Run each structured query against the full text of a single document.
    Uses the LLM directly (not RAG retriever) for document-level analysis.
    """
    results = {"filename": filename}
    
    for field, question in queries.items():
        prompt = ChatPromptTemplate.from_template(
            "You are an expert document analyst.\n\n"
            "Document content:\n{content}\n\n"
            "Task: {question}\n\n"
            "Answer concisely and factually based solely on the document above."
        )
        chain = LLMChain(llm=llm, prompt=prompt)
        answer = chain.run(content=file_content[:8000], question=question)  # trim to token budget
        results[field] = answer.strip()
    
    return results


# ── Group raw pages by filename ────────────────────────────────────────────
from collections import defaultdict

docs_by_file: dict[str, str] = defaultdict(str)
for doc in raw_docs:
    fname = doc.metadata.get("filename", "unknown")
    docs_by_file[fname] += doc.page_content + "\n"

print(f"Documents to analyse: {list(docs_by_file.keys())}\n")

# ── Run analysis ───────────────────────────────────────────────────────────
analysis_results = []

for fname, content in docs_by_file.items():
    print(f"Analysing: {fname} ...")
    row = analyse_document(fname, content, ANALYSIS_QUERIES)
    row["total_pages"]  = sum(1 for d in raw_docs if d.metadata.get("filename") == fname)
    row["total_chunks"] = sum(1 for d in chunked_docs if d.metadata.get("filename") == fname)
    row["total_chars"]  = len(content)
    analysis_results.append(row)
    print(f"  ✓ {fname} done")

print(f"\n✅ Analysis complete.  {len(analysis_results)} document(s) processed.")

## Step 8 — Export Results to CSV

Flatten the analysis dictionary into a `pandas` DataFrame and write to `output/analysis_results.csv`.

In [ ]:
import pandas as pd

# Build DataFrame — each row is one PDF document
results_df = pd.DataFrame(analysis_results)

# Reorder columns for readability
ordered_cols = (
    ["filename", "total_pages", "total_chunks", "total_chars"]
    + list(ANALYSIS_QUERIES.keys())
)
results_df = results_df[[c for c in ordered_cols if c in results_df.columns]]

# Save to CSV (UTF-8 with BOM for Excel compatibility)
results_df.to_csv(CSV_OUTPUT, index=False, encoding="utf-8-sig")

print(f"CSV saved → {CSV_OUTPUT.resolve()}")
print(f"Rows: {len(results_df)}  |  Columns: {list(results_df.columns)}\n")

# Preview
results_df.head()

## Step 9 — Visualisations with Plotly

Six interactive charts tailored to the Software Solution Architecture curriculum:
1. **Document Size Comparison** — pages & chunks per PDF
2. **Difficulty Level Distribution** — Introductory / Intermediate / Advanced breakdown
3. **Module Topics** — primary topic per document as a treemap
4. **Top Architecture Patterns** — frequency of patterns across all documents
5. **Top Technologies & Tools** — frequency of tech/tools across all documents
6. **Document Similarity Heatmap** — cosine similarity between document embeddings

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import numpy as np
import pandas as pd

# Ensure plotly renders correctly inside Jupyter
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# ── Load results: prefer CSV on disk, fall back to in-memory DataFrame ─────
if CSV_OUTPUT.exists():
    results_df = pd.read_csv(CSV_OUTPUT, encoding="utf-8-sig")
    print(f"✅ Loaded results from CSV: {CSV_OUTPUT.resolve()}  ({len(results_df)} row(s))")
elif "results_df" in dir() and results_df is not None:
    print("⚠️  CSV not found — using in-memory DataFrame from Step 8.")
else:
    raise RuntimeError(
        f"No data available. Run Step 7–8 first, or ensure '{CSV_OUTPUT}' exists."
    )

print(f"   Columns: {list(results_df.columns)}\n")

# ── Helper: split comma-separated column into a frequency Series ────────────
def explode_csv_col(df: pd.DataFrame, col: str, top_n: int = 20) -> pd.DataFrame:
    items = []
    for cell in df[col].dropna():
        items.extend([v.strip() for v in str(cell).split(",") if v.strip()])
    freq = pd.Series(items).value_counts().head(top_n).reset_index()
    freq.columns = [col, "Frequency"]
    return freq


# ── Chart 1: Document Size (pages + chunks) ─────────────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Pages per Document", "Chunks per Document"]
)
fig1.add_trace(
    go.Bar(x=results_df["filename"], y=results_df["total_pages"],
           name="Pages", marker_color="#4C78A8"),
    row=1, col=1
)
fig1.add_trace(
    go.Bar(x=results_df["filename"], y=results_df["total_chunks"],
           name="Chunks", marker_color="#72B7B2"),
    row=1, col=2
)
fig1.update_layout(title_text="Document Size Comparison", showlegend=False, height=420)
display(fig1)


# ── Chart 2: Difficulty Level Distribution ──────────────────────────────────
if "difficulty_level" in results_df.columns:
    diff_counts = (
        results_df["difficulty_level"]
        .str.strip()
        .str.capitalize()
        .value_counts()
        .reindex(["Introductory", "Intermediate", "Advanced"], fill_value=0)
        .reset_index()
    )
    diff_counts.columns = ["Level", "Count"]

    colour_map = {"Introductory": "#2ECC71", "Intermediate": "#F39C12", "Advanced": "#E74C3C"}
    fig2 = px.bar(
        diff_counts,
        x="Level", y="Count",
        color="Level",
        color_discrete_map=colour_map,
        title="Difficulty Level Distribution",
        text="Count",
    )
    fig2.update_traces(textposition="outside")
    fig2.update_layout(height=400, showlegend=False)
    display(fig2)


# ── Chart 3: Module Topics — Documents per Topic ───────────────────────────
if "module_topic" in results_df.columns:
    topics_df = (
        results_df[["filename", "module_topic"]]
        .dropna()
        .assign(module_topic=lambda d: d["module_topic"].str.strip())
    )

    # Aggregate: count documents per topic, collect filenames for hover
    topic_agg = (
        topics_df
        .groupby("module_topic", sort=False)
        .agg(
            count=("filename", "count"),
            documents=("filename", lambda x: "<br>".join(x.tolist())),
        )
        .reset_index()
        .sort_values("count", ascending=True)
    )

    fig3 = px.bar(
        topic_agg,
        x="count",
        y="module_topic",
        orientation="h",
        title="Documents per Module Topic",
        text="count",
        color="count",
        color_continuous_scale="Blues",
        labels={"module_topic": "Topic", "count": "# Documents"},
        custom_data=["documents"],
    )
    fig3.update_traces(
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Documents: %{x}<br><br>%{customdata[0]}<extra></extra>",
    )
    fig3.update_layout(height=max(400, len(topic_agg) * 45), showlegend=False,
                       coloraxis_showscale=False)
    display(fig3)


# ── Chart 4: Top Architecture Patterns ──────────────────────────────────────
if "architecture_patterns" in results_df.columns:
    patterns_freq = explode_csv_col(results_df, "architecture_patterns", top_n=20)
    fig4 = px.bar(
        patterns_freq,
        x="Frequency", y="architecture_patterns",
        orientation="h",
        title="Top Architecture Patterns Across All Documents",
        color="Frequency",
        color_continuous_scale="Blues",
        labels={"architecture_patterns": "Pattern"},
    )
    fig4.update_layout(height=520, yaxis={"autorange": "reversed"})
    display(fig4)


# ── Chart 5: Top Technologies & Tools ───────────────────────────────────────
if "technologies_tools" in results_df.columns:
    tech_freq = explode_csv_col(results_df, "technologies_tools", top_n=20)
    fig5 = px.bar(
        tech_freq,
        x="Frequency", y="technologies_tools",
        orientation="h",
        title="Top Technologies & Tools Across All Documents",
        color="Frequency",
        color_continuous_scale="Teal",
        labels={"technologies_tools": "Technology / Tool"},
    )
    fig5.update_layout(height=520, yaxis={"autorange": "reversed"})
    display(fig5)


# ── Chart 6: Document Embedding Similarity Heatmap ──────────────────────────
if "docs_by_file" in dir() and len(docs_by_file) > 1:
    doc_names  = list(docs_by_file.keys())
    doc_texts  = [docs_by_file[n][:4000] for n in doc_names]

    doc_vectors = np.array(embedding.embed_documents(doc_texts))
    norms       = np.linalg.norm(doc_vectors, axis=1, keepdims=True)
    sim_matrix  = (doc_vectors @ doc_vectors.T) / (norms @ norms.T + 1e-9)

    fig6 = px.imshow(
        sim_matrix,
        x=doc_names, y=doc_names,
        text_auto=".2f",
        color_continuous_scale="RdBu",
        title="Document Similarity Heatmap (Cosine)",
        zmin=0, zmax=1,
    )
    fig6.update_layout(height=520)
    display(fig6)
else:
    print("ℹ️  Similarity heatmap skipped — requires ≥ 2 documents and docs_by_file in memory.")

print("\n✅ All charts rendered.")


## Step 10 — HTML Report Generation

Combine the analysis table and all Plotly charts into a single, self-contained `report.html`
using a Jinja2 template.  The file can be opened directly in any browser — no server needed.

Sections in the report:
- Header with generation timestamp
- Executive Summary table (from CSV)
- Per-document detail cards (summary, topics, entities, recommendations)
- All 4 interactive Plotly charts embedded as inline JavaScript

In [ ]:
from jinja2 import Template
from datetime import datetime
import plotly.io as pio

# ── Serialise charts to self-contained HTML fragments ─────────────────────
def fig_to_html(fig, div_id: str) -> str:
    return pio.to_html(fig, full_html=False, div_id=div_id, include_plotlyjs="cdn")

charts_html = {}
charts_html["size_chart"]       = fig_to_html(fig1, "chart_size")
if "difficulty_level" in results_df.columns:
    charts_html["difficulty_chart"] = fig_to_html(fig2, "chart_difficulty")
if "module_topic" in results_df.columns:
    charts_html["topics_chart"]     = fig_to_html(fig3, "chart_topics")
if "architecture_patterns" in results_df.columns:
    charts_html["patterns_chart"]   = fig_to_html(fig4, "chart_patterns")
if "technologies_tools" in results_df.columns:
    charts_html["tech_chart"]       = fig_to_html(fig5, "chart_tech")
if "docs_by_file" in dir() and len(docs_by_file) > 1:
    charts_html["similarity_chart"] = fig_to_html(fig6, "chart_similarity")

# ── Jinja2 HTML template ───────────────────────────────────────────────────
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Software Solution Architecture — Learning Materials Analysis</title>
  <style>
    body      { font-family: 'Segoe UI', Arial, sans-serif; margin: 0; background: #f5f7fa; color: #333; }
    header    { background: #1a3a5c; color: white; padding: 24px 40px; }
    header h1 { margin: 0; font-size: 1.8em; }
    header p  { margin: 4px 0 0; opacity: 0.75; font-size: 0.9em; }
    main      { max-width: 1280px; margin: 30px auto; padding: 0 24px; }
    section   { background: white; border-radius: 8px; padding: 24px 28px; margin-bottom: 24px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.08); }
    h2        { color: #1a3a5c; border-bottom: 2px solid #e3eaf3; padding-bottom: 8px; }
    h3        { color: #2c5f9e; margin-top: 20px; }
    table     { width: 100%; border-collapse: collapse; font-size: 0.85em; }
    th        { background: #1a3a5c; color: white; padding: 10px 12px; text-align: left; }
    td        { padding: 8px 12px; border-bottom: 1px solid #e8edf3; vertical-align: top; }
    tr:hover td { background: #f0f5ff; }
    .doc-card { border-left: 4px solid #2c5f9e; padding-left: 16px; margin-bottom: 28px; }
    .label    { font-weight: 600; color: #555; min-width: 200px; display: inline-block; vertical-align: top; }
    .tag      { display: inline-block; background: #e3eaf3; border-radius: 12px;
                padding: 2px 10px; margin: 2px 2px 4px; font-size: 0.8em; }
    .tag.pattern  { background: #dce8f5; color: #1a3a5c; }
    .tag.tech     { background: #d5f0e8; color: #1a5c3a; }
    .tag.concept  { background: #f5e8d5; color: #5c3a1a; }
    .badge        { display: inline-block; border-radius: 4px; padding: 2px 10px;
                    font-size: 0.8em; font-weight: 600; color: white; }
    .badge.Introductory { background: #2ECC71; }
    .badge.Intermediate { background: #F39C12; }
    .badge.Advanced     { background: #E74C3C; }
    footer    { text-align: center; padding: 20px; color: #888; font-size: 0.8em; }
  </style>
</head>
<body>
<header>
  <h1>Software Solution Architecture — Learning Materials Analysis</h1>
  <p>Generated: {{ generated_at }} &nbsp;|&nbsp; Documents: {{ doc_count }} &nbsp;|&nbsp; Model: {{ llm_info }}</p>
</header>
<main>

  <section>
    <h2>Executive Summary</h2>
    {{ summary_table }}
  </section>

  <section>
    <h2>Document Details</h2>
    {% for row in rows %}
    <div class="doc-card">
      <h3>{{ row.filename }}</h3>
      <p>
        <span class="label">Topic:</span> {{ row.get('module_topic', '—') }}
        &nbsp;&nbsp;
        <span class="label">Difficulty:</span>
        <span class="badge {{ row.get('difficulty_level','').strip().capitalize() }}">
          {{ row.get('difficulty_level', '—') }}
        </span>
        &nbsp;&nbsp;
        <span class="label">Pages / Chunks:</span> {{ row.get('total_pages','?') }} / {{ row.get('total_chunks','?') }}
      </p>
      <p><span class="label">Summary:</span> {{ row.get('summary', '—') }}</p>
      <p><span class="label">Learning Objectives:</span> {{ row.get('learning_objectives', '—') }}</p>
      <p><span class="label">Architecture Patterns:</span><br>
        {% for t in row.get('architecture_patterns','').split(',') %}
          {% if t.strip() %}<span class="tag pattern">{{ t.strip() }}</span>{% endif %}
        {% endfor %}
      </p>
      <p><span class="label">Technologies & Tools:</span><br>
        {% for t in row.get('technologies_tools','').split(',') %}
          {% if t.strip() %}<span class="tag tech">{{ t.strip() }}</span>{% endif %}
        {% endfor %}
      </p>
      <p><span class="label">Key Concepts:</span><br>
        {% for t in row.get('key_concepts','').split(',') %}
          {% if t.strip() %}<span class="tag concept">{{ t.strip() }}</span>{% endif %}
        {% endfor %}
      </p>
      <p><span class="label">Architectural Decisions:</span> {{ row.get('architectural_decisions', '—') }}</p>
      <p><span class="label">Prerequisites:</span> {{ row.get('prerequisites', '—') }}</p>
      <p><span class="label">Exercises / Labs:</span> {{ row.get('exercises_labs', '—') }}</p>
      <p><span class="label">Gaps & Recommendations:</span> {{ row.get('gaps_recommendations', '—') }}</p>
    </div>
    {% endfor %}
  </section>

  <section>
    <h2>Visualisations</h2>
    {{ charts_html.get('size_chart', '') }}
    {{ charts_html.get('difficulty_chart', '') }}
    {{ charts_html.get('topics_chart', '') }}
    {{ charts_html.get('patterns_chart', '') }}
    {{ charts_html.get('tech_chart', '') }}
    {{ charts_html.get('similarity_chart', '') }}
  </section>

</main>
<footer>RAG Pipeline • LangChain + ChromaDB + Plotly &nbsp;|&nbsp; Software Solution Architecture Program</footer>
</body>
</html>
"""

# ── Render with data ───────────────────────────────────────────────────────
llm_info = f"{LLM_PROVIDER} / {OPENAI_MODEL if LLM_PROVIDER == 'openai' else COHERE_MODEL}"

summary_cols = ["filename", "module_topic", "difficulty_level", "total_pages", "total_chunks"]
summary_cols = [c for c in summary_cols if c in results_df.columns]

summary_html = results_df[summary_cols].to_html(
    index=False, escape=True, border=0, classes="",
)

template    = Template(HTML_TEMPLATE)
html_output = template.render(
    generated_at  = datetime.now().strftime("%Y-%m-%d %H:%M"),
    doc_count     = len(results_df),
    llm_info      = llm_info,
    summary_table = summary_html,
    rows          = results_df.to_dict(orient="records"),
    charts_html   = charts_html,
)

HTML_REPORT.write_text(html_output, encoding="utf-8")
print(f"HTML report saved → {HTML_REPORT.resolve()}")

from IPython.display import IFrame
IFrame(src=str(HTML_REPORT.resolve()), width="100%", height=650)

## Bonus — Interactive Q&A

Ask any question across all loaded PDFs.  The chain uses conversation memory,
so follow-up questions work naturally.

In [ ]:
def ask(question: str, show_sources: bool = True) -> str:
    """
    Query the RAG chain and display the answer + source document snippets.
    Conversation history is preserved across calls within this session.
    """
    result = qa_chain({"question": question, "chat_history": []})
    answer = result["answer"]
    
    print(f"\n💬 Q: {question}")
    print(f"\n🤖 A: {answer}")
    
    if show_sources and result.get("source_documents"):
        print("\n📄 Sources:")
        for i, doc in enumerate(result["source_documents"], 1):
            fname = doc.metadata.get("filename", "unknown")
            page  = doc.metadata.get("page", "?")
            print(f"  [{i}] {fname}  (page {page})")
            print(f"      {doc.page_content[:200].strip()}…")
    
    return answer


# ── Change the question below and re-run this cell ─────────────────────
USER_QUESTION = "What are the main topics covered across all documents?"

answer = ask(USER_QUESTION)